In [1]:
from sklearn.linear_model import LogisticRegression, LinearRegression
import pandas as pd

In [2]:
df = pd.read_csv("WineQT.csv")

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1143 entries, 0 to 1142
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1143 non-null   float64
 1   volatile acidity      1143 non-null   float64
 2   citric acid           1143 non-null   float64
 3   residual sugar        1143 non-null   float64
 4   chlorides             1143 non-null   float64
 5   free sulfur dioxide   1143 non-null   float64
 6   total sulfur dioxide  1143 non-null   float64
 7   density               1143 non-null   float64
 8   pH                    1143 non-null   float64
 9   sulphates             1143 non-null   float64
 10  alcohol               1143 non-null   float64
 11  quality               1143 non-null   int64  
 12  Id                    1143 non-null   int64  
dtypes: float64(11), int64(2)
memory usage: 116.2 KB


In [4]:
correlation = df.corr()['quality']
print(correlation)

fixed acidity           0.121970
volatile acidity       -0.407394
citric acid             0.240821
residual sugar          0.022002
chlorides              -0.124085
free sulfur dioxide    -0.063260
total sulfur dioxide   -0.183339
density                -0.175208
pH                     -0.052453
sulphates               0.257710
alcohol                 0.484866
quality                 1.000000
Id                      0.069708
Name: quality, dtype: float64


In [14]:
df['quality'].value_counts().sort_index()

quality
3      6
4     33
5    483
6    462
7    143
8     16
Name: count, dtype: int64

In [118]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif

X = df.drop(columns = ['quality', 'Id'])
y = df['quality']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [60]:
from sklearn.metrics import accuracy_score, f1_score

model = LogisticRegression(
    solver='saga',
    class_weight='balanced',
    max_iter=5000,
    random_state=42
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1_weighted = f1_score(y_test, y_pred, average='weighted')
print(accuracy, f1_weighted)

0.4978165938864629 0.5348491512276321


In [51]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, y_pred))

[[ 0  0  1  0  0  0]
 [ 0  0  4  3  0  0]
 [ 0  1 74 20  2  0]
 [ 0  1 22 58 10  1]
 [ 0  0  2 14 13  0]
 [ 0  0  0  1  2  0]]


In [105]:
from sklearn.svm import SVC, LinearSVC

svc_model = SVC(
        C=20,
        kernel='rbf',
        gamma='scale',
        random_state=42)

svc_model.fit(X_train, y_train)
svc_pred = svc_model.predict(X_test)

svc_acc = accuracy_score(y_test, svc_pred)
svc_f1_weighted = f1_score(y_test, svc_pred, average='weighted')
print(svc_acc, svc_f1_weighted)

0.6855895196506551 0.6820756813848126


In [139]:
import numpy as np
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model2 = Lasso(alpha=0.001)
# model2 = LinearRegression()

model2.fit(X_train, y_train)
lin_pred = np.round(model2.predict(X_test))
lin_pred = np.clip(lin_pred, y.min(), y.max()).astype(int)

mae = mean_absolute_error(y_test, lin_pred)
mse = mean_squared_error(y_test, lin_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, lin_pred)
print(mae, mse, rmse, r2)

0.388646288209607 0.4497816593886463 0.6706576320214707 0.19172777739702596


In [155]:
from sklearn.svm import SVR

svr_model = SVR(
    kernel='rbf',
    C=3,
    gamma='scale')

svr_model.fit(X_train, y_train)
svr_pred = svr_model.predict(X_test)

mae = mean_absolute_error(y_test, svr_pred)
rmse = np.sqrt(mean_squared_error(y_test, svr_pred))
r2 = r2_score(y_test, svr_pred)
print(mae, rmse, r2)

0.3960255673637322 0.5438193506094828 0.46854655642418175


реализация линейного SVM

In [168]:
y_binary = np.where(y <= 5, -1, 1)

class LinearSVM:

    def __init__(self, lr = 0.001, lambda_param = 0.01, n_iters = 1000):
        self.lr = lr
        self.lambda_param = lambda_param
        self.n_iters = n_iters
        self.w = None
        self.b = None

    def fit(self, X, y):
        n_samples, n_features = X.shape

        self.w = np.zeros(n_features)
        self.b = 0

        for i in range(self.n_iters):
            for idx, x_i in enumerate(X):
                condition = y[idx] * (np.dot(x_i, self.w) + self.b)

                if condition >= 1:
                    self.w -= self.lr * (2 * self.lambda_param * self.w)
                else:
                    self.w -= self.lr * (2 * self.lambda_param * self.w - y[idx] * x_i)
                    self.b -= self.lr * (-y[idx])

    def predict(self, X):
        linear_output = np.dot(X, self.w) + self.b
        return np.sign(linear_output)

In [169]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif

X = df.drop(columns = ['quality', 'Id'])
y = df['quality']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

y_train = np.where(y_train <= 5, -1, 1)
y_test = np.where(y_test <= 5, -1, 1)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [170]:
model = LinearSVM(
    lr=0.001,
    lambda_param=0.01,
    n_iters=1000
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [171]:
from sklearn.metrics import accuracy_score, f1_score

acc = accuracy_score(y_test,y_pred)
f1 = f1_score(y_test,y_pred)
print(acc, f1)

0.7510917030567685 0.7710843373493976
